In [0]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

CODE_ROOT = Path(
    "/Workspace/Users/arfaashaikh101@gmail.com/retail_dev_dbx"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from scripts.bronze_loader import load_config
from scripts.control import setup_control
from scripts.layer_runner import run_transform_layer

config = load_config(
    spark,
    CODE_ROOT / "config/source_contracts.yaml",
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

control_config = setup_control(
    spark,
    config["namespace"],
    CODE_ROOT / "config/control_contracts.yaml",
)

In [0]:
gold_run_id, result = run_transform_layer(
    spark=spark,
    config=config,
    control_config=control_config,
    layer="GOLD",
    trigger_type="MANUAL",
)

print("Gold run:", gold_run_id)
print(result)

In [0]:
display(
    spark.table(control_config["tables"]["master"])
    .filter(F.col("run_id") == gold_run_id)
    .select(
        "control_id",
        "target_table",
        "final_status",
        "write_outcome",
        "read_count",
        "committed_count",
        "target_delta_version",
        "failure_reason",
    )
)